# TorchAX Training Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_training_quickstart.ipynb)

Fine-tune a HuggingFace model on TPU with LoRA in under 10 cells.

**Full tutorial:** [torchax_training_tutorial.ipynb](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_training_tutorial.ipynb) | [Blog post](https://dev.to/ahmed_elnaggar/fine-tune-any-huggingface-model-on-tpus-with-torchax)

In [ ]:
# Install dependencies (auto-detect TPU/GPU)
import subprocess
try:
    subprocess.run(["ls", "/dev/accel0"], capture_output=True, check=True)
    accel = "tpu"
except Exception:
    try:
        subprocess.run(["nvidia-smi"], capture_output=True, check=True)
        accel = "gpu"
    except Exception:
        accel = "cpu"

!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
if accel == "tpu":
    !pip install -q -U jax[tpu]
elif accel == "gpu":
    !pip install -q -U jax[cuda12]
else:
    !pip install -q -U jax
!pip install -q -U torchax transformers flax peft datasets optax
print(f"Accelerator: {accel}")

In [ ]:
# Configuration — change these to customize
MODEL_NAME = "google/gemma-4-E2B-it"
DATASET_NAME = "databricks/databricks-dolly-15k"
TRAINING_MODE = "lora"      # "lora" or "full"
LORA_RANK = 16
LORA_ALPHA = 32
MAX_SEQ_LEN = 512
MAX_TRAIN_SAMPLES = 1000
BATCH_SIZE = 2
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
SEED = 42

print(f"Mode: {TRAINING_MODE} | Model: {MODEL_NAME}")

In [ ]:
# Load dataset and tokenizer
import os, tempfile
import datasets as hf_datasets
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

raw = hf_datasets.load_dataset(DATASET_NAME, split="train").shuffle(seed=SEED).select(range(MAX_TRAIN_SAMPLES))

def tokenize(example):
    user_content = example["instruction"]
    if example.get("context", ""):
        user_content += f"\n\nContext: {example['context']}"
    messages = [{"role": "user", "content": user_content}, {"role": "assistant", "content": example["response"]}]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return tokenizer(text, padding="max_length", max_length=MAX_SEQ_LEN, truncation=True, return_tensors=None)

tokenized = raw.map(tokenize, load_from_cache_file=False,
                     cache_file_name=os.path.join(tempfile.gettempdir(), "qs_cache.arrow"),
                     remove_columns=raw.column_names)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
train_dataloader = DataLoader(tokenized, shuffle=True, collate_fn=collator, batch_size=BATCH_SIZE)
print(f"Loaded {len(tokenized)} examples, {len(train_dataloader)} batches")

In [ ]:
# Load model + apply LoRA
import torch
import torchax as tx
import peft

with tx.disable_temporarily():
    model = __import__("transformers").AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

if TRAINING_MODE == "lora":
    model = peft.get_peft_model(model, peft.LoraConfig(
        task_type=peft.TaskType.CAUSAL_LM, inference_mode=False,
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
    ))
    model.print_trainable_parameters()
else:
    print(f"Full fine-tuning: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

# Move to JAX
tx.enable_globally()
device = torch.device("jax")
model.to(device)
model.train()
print(f"Model on {device}")

In [ ]:
# Set up functional training
import numpy as np
import optax
import torchax.train

params = {n: p for n, p in model.named_parameters() if p.requires_grad}
buffers = dict(model.named_buffers())
frozen = {n: p for n, p in model.named_parameters() if not p.requires_grad}
buffers.update(frozen)

if not params:
    raise RuntimeError("No trainable params found! Check LoRA target_modules.")

optimizer = optax.adamw(learning_rate=LEARNING_RATE, weight_decay=0.01)
opt_state = tx.interop.call_jax(optimizer.init, params)

def model_fn(weights, buffers, batch):
    return torch.func.functional_call(model, {**weights, **buffers}, args=(), kwargs=batch)

def loss_fn(output, labels):
    return output.loss

step_fn = tx.train.make_train_step(model_fn, loss_fn, optimizer)
print(f"Training setup: {sum(p.numel() for p in params.values()) / 1e6:.2f}M trainable params")

In [ ]:
# Training loop
import time
from tqdm.auto import tqdm

torch.manual_seed(SEED)
train_losses = []
start = time.time()

for epoch in range(NUM_EPOCHS):
    pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}")
    for step, batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss, params, opt_state = step_fn(params, buffers, opt_state, batch, batch["labels"])
        train_losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

elapsed = time.time() - start
print(f"\nDone! {len(train_losses)} steps in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Final loss: {train_losses[-1]:.4f}")

In [ ]:
# Save fine-tuned model
save_dir = "./fine_tuned_model"

with torch.no_grad():
    cpu_state_dict = {
        name: torch.tensor(np.array(p)).contiguous()
        for name, p in params.items()
    }
    model.save_pretrained(save_dir, state_dict=cpu_state_dict)
tokenizer.save_pretrained(save_dir)

total_size = sum(os.path.getsize(os.path.join(save_dir, f)) for f in os.listdir(save_dir))
print(f"Saved to {save_dir} ({total_size / 1e6:.1f} MB)")

In [ ]:
# Reload and test
import gc
for v in ['params', 'buffers', 'opt_state', 'frozen']:
    if v in dir():
        del globals()[v]
gc.collect()

with tx.disable_temporarily():
    if TRAINING_MODE == "lora":
        reloaded = __import__("transformers").AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
        reloaded = peft.PeftModel.from_pretrained(reloaded, save_dir)
    else:
        reloaded = __import__("transformers").AutoModelForCausalLM.from_pretrained(save_dir, torch_dtype=torch.bfloat16)

reloaded.to(device).eval()

def generate(instruction, max_new_tokens=80):
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": instruction}], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    generated = ids.clone()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = reloaded(generated, use_cache=False)
            tok = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
            generated = torch.cat([generated, tok], dim=-1)
            if tok.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0, ids.shape[1]:], skip_special_tokens=True)

for q in ["Explain machine learning in simple terms.", "Write a haiku about coding."]:
    print(f"Q: {q}")
    print(f"A: {generate(q)}\n")

## Want to learn more?

- [Full training tutorial](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_training_tutorial.ipynb) — detailed explanations, full fine-tuning, evaluation, visualizations
- [Inference tutorial](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_huggingface_tutorial.ipynb) — JIT compilation, benchmarks, text generation
- [Blog post](https://dev.to/ahmed_elnaggar/fine-tune-any-huggingface-model-on-tpus-with-torchax) — the complete written tutorial
- [torchax GitHub](https://github.com/google/torchax) — library source
- [PEFT LoRA example](https://github.com/google/torchax/blob/main/examples/peft_lora_training.py) — official torchax training example